In this notebook we generate related links for pages using functional data (what people view on GOV.UK), and LLR (the Log-Likelihood Ratio).

LLR (log likelihood ratio) looks at how often pages A and B are viewed together, but also how often in general each page is viewed (if page A co-occurs at similar rates with B and C, but C is less popular sitewide, it is seen as a more relevant recommendation for A).

A blog about LLR: http://tdunning.blogspot.com/2008/03/surprise-and-coincidence.html
Paper: http://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.14.5962&rep=rep1&type=pdf


Some notes on what we do:
- LLR by content ID
- look at page hits sequentially - link page viewed AFTER page
- filter out "finding' pages - `document_type` in ['document_collection', 'finder', 'homepage', 'license_finder', 'mainstream_browse_page', 'organisation', 'search', 'service_manual_homepage', 'service_manual_topic', 'services_and_information', 'taxon', 'topic', 'topical_event']
- filter out 'fatality_notice', 'contact', 'service_sign_in','html_publication', 'calculator', 'completed_transaction' `document_type`s
- filter out embedded links that already exist on pages

# Setup

In [ ]:
import os 
import math
from ast import literal_eval
import pandas as pd
import json

In [ ]:
# set up pandas, bigquery, data dir stuff
ProjectID = 'govuk-bigquery-analytics'
KEY_DIR = os.getenv("BQ_KEY_DIR")
key_file_path = os.path.join(KEY_DIR, os.listdir(KEY_DIR)[0])

DATA_DIR = os.getenv("DATA_DIR")

# Run LLR Query
Node2Vec used 3 weeks of data, so we should use 3 weeks here to give it the same chance of capturing the functional network

The query dequencs pageviews using hitNumber, so we look at pages viewed *after* other pages

You have to sub in `START_DATE` and `END_DATE`

In [ ]:
file_object  = open('../queries/llr_pages_content_id_split_up_hitNumber.sql', 'r')
QUERY = file_object.read()

In [ ]:
print(QUERY)

In [ ]:
print(QUERY.format(START_DATE='20190222', END_DATE='20190314'))

In [ ]:
df_llr_recs = pd.io.gbq.read_gbq(
    QUERY.format(START_DATE='20190222', END_DATE='20190314'),
                           project_id=ProjectID,
                           reauth=False,
                           verbose=True,
                           private_key=key_file_path,
                           dialect='standard')

In [ ]:
df_llr_recs.to_csv(os.path.join(DATA_DIR, 'llr', 'llr_recs_20190222_20190314.csv.gz'),
                     compression='gzip', index=False)

In [ ]:
# df_llr_recs = pd.read_csv(os.path.join(DATA_DIR, 'llr', 'llr_recs_20190222_20190314.csv.gz'),
#                      compression='gzip')

In [ ]:
df_llr_recs.head()

Quite a big DF at the moment, as I've taken up to 100 links for each page, only because I don't know how many will be filtered out by removing embedded links. I've also looked at pages over 3 weeks, so the number of occurrences for each page will be fairly high

In [ ]:
print(df_llr_recs.shape)
print(df_llr_recs[df_llr_recs['co_occurrences']>1].shape)
print(df_llr_recs[df_llr_recs['co_occurrences']>10].shape)
print(df_llr_recs[df_llr_recs['co_occurrences']>100].shape)

In [ ]:
df_llr_recs['page_1'].nunique()

# Get top n related links, without embedded links
Filter out embedded links that already exist in pages, and get the top n related links for each page

## Get embedded links
Get the embedded links for each page from `content_json_extended.csv.gz` (created in [govuk-network-embedding](https://github.com/alphagov/govuk-network-embedding/blob/2nd-iteration/notebooks/data_extract/extract_links_from_json.ipynb) )

In [ ]:
# get previously extracted content API data
content_json_file = os.path.join(DATA_DIR, 'content_api', "11-02-19",
                                 "content_json_extended.csv.gz")
content = pd.read_csv(content_json_file, compression="gzip")

In [ ]:
content.columns

In [ ]:
# dictionary of base paths to content IDs
content_base_cid = dict(zip(content.base_path, content.content_id))

In [ ]:
content['embedded_links'] = content['embedded_links'].map(literal_eval)

# for all the embedded link URLs in a row, get their content IDs if they are in content_base_cid
content['embedded_cids'] = content['embedded_links'].map(lambda x: [content_base_cid[l] for l in x 
                                                                    if l in content_base_cid.keys()])

In [ ]:
# dictionary of content IDs and the content IDs of the embedded links in them
cid_link_cids = dict(zip(content.content_id, content.embedded_cids))

In [ ]:
# dictionary of content IDs and the content IDs of the embedded links in them,
# only if there are embedded links
cid_link_cids_with_emb = {k: v for k, v in cid_link_cids.items() if len(v) > 0}

In [ ]:
len(cid_link_cids_with_emb.keys())

## Get titles and base paths
For each content ID get its title and base_path from `content_json_extended.csv.gz` so that humans can assess our links

In [ ]:
cids_titles = dict(zip(content.content_id, content.title))

In [ ]:
cids_paths = dict(zip(content.content_id, content.base_path))

Create a dictionary `related_links` from `df_llr_recs`, the content IDs of each page are the keys, the values are arrays of the recommended links, with LLR scores, ranks fro the SQL, and co_occurrence counts

In [ ]:
df_llr_recs.columns

In [ ]:
related_links = {}

for row in df_llr_recs.itertuples():
    if row.page_1 not in related_links:
        related_links[row.page_1] = list()
    related_links[row.page_1].append (
        {'link': row.page_2, 'llr_score': row.llr_score, 'rank': row.rank, 
         'co_occurrences': row.co_occurrences})

In [ ]:
def get_page_title(content_id):
    try:
        return cids_titles[content_id]
    except KeyError:
        return 'unknown'

def get_page_url(content_id):
    try:
        return f"www.gov.uk{cids_paths[content_id]}"
    except KeyError:
        return 'unknown'

## Remove embedded links
With a dictionary of content IDs : related links, and a dictionary of content IDs : embedded links, filter out all embedded links that occur in related links for each content ID

In [ ]:
def remove_embedded(related_links_dict, cid_link_cids_with_emb):
    for content_id, embedded_links in cid_link_cids_with_emb.items():
        for emb_link in embedded_links:
            try:
                related_links_dict[content_id] = [
                    recommended_page
                    for recommended_page in related_links_dict[content_id] 
                    if recommended_page['link'] != emb_link]
            except KeyError:
                continue

In [ ]:
remove_embedded(related_links, cid_link_cids_with_emb)

## Remove links for some excluded pages
We don't want to generate related links for some pages, so remove them here

In [ ]:
PAGES_TO_EXCLUDE = [
    # https://www.gov.uk/contact-student-finance-england
    'd490be5f-1998-4f20-ab52-d3dd5db7fa71',
    # https://www.gov.uk/student-finance-forms
    '37e27ec1-ef3e-4b3c-a2f6-14dc42c4c162',
    # https://www.gov.uk/funding-for-postgraduate-study
    'bcd1365f-8496-40f5-b9cf-06f2493e48c4',
    # https://www.gov.uk/advanced-learner-loan
    '23eee5eb-7e24-4a7f-bf92-112f8c8132bc',
    # https://www.gov.uk/teacher-training-funding
    '708334c4-2855-4d45-b311-72a26b03529a',
    # https://www.gov.uk/extra-money-pay-university
    'eff0d788-3b5e-4090-8b56-aaa0a4bf3f25',
    # https://www.gov.uk/nhs-bursaries
    '6a131bf3-6d52-4512-98cc-348c243fba8f',
    # https://www.gov.uk/travel-grants-students-england
    '05aaf43f-7555-46b3-99e5-a34e48b0eec9',
    # https://www.gov.uk/career-development-loans
    'cd69a882-474a-4dd9-9689-adde2fcd618c',
    # https://www.gov.uk/care-to-learn
    'c1873350-0c82-469d-9f59-4658be95fdd2',
    # https://www.gov.uk/dance-drama-awards
    '04d10810-8620-4edd-a06d-f5ecfc199414',
    # https://www.gov.uk/social-work-bursaries
    'eeeee6fc-5f61-4b8a-bb41-efe12bad9e2c',
    # https://www.gov.uk/student-finance-if-you-suspend-or-leave
    '2585264f-9562-4d4b-bbd0-4dae7b7bc2b1']

In [ ]:
def remove_links_from_pages(related_links_dict, pages_to_exclude):
    for content_id in pages_to_exclude:
        try:
            del related_links_dict[content_id]
        except KeyError:
            continue

In [ ]:
remove_links_from_pages(related_links, PAGES_TO_EXCLUDE)

In [ ]:
len(related_links)

## Get top n, add titles and base paths
Using a clean dictionary of related links (no replication of embedded links), iterate through the dictionary:
 - sort the recommended related links by their rank, and take the top n
 - add titles and base paths for the pages and their related links
 - return a DataFrame with all this info

In [ ]:
def get_top_n(related_links_dict, n=10):
    pages_links = []
    for page, recs in related_links_dict.items():
        title = get_page_title(page)
        base_path = get_page_url(page)
        sorted_recs = sorted(recs, key=lambda k: k['rank'])[:n]
        for rec in sorted_recs:
            page_link = {"title": title,
                         "link_title": get_page_title(rec['link']),
                         "base_path": base_path,
                         "link_base_path": get_page_url(rec['link']),
                         "page": page}
            page_link.update(rec)
            pages_links.append(page_link)
    return pd.DataFrame(pages_links)

In [ ]:
top_5_df = get_top_n(related_links, n=5)

In [ ]:
top_5_df

In [ ]:
top_5_df.shape

In [ ]:
top_5_df.query('base_path != "unknown"').shape

# Get related links for colleagues to review

## Fix LLR score as we forgot to divide it by N
I've updated the SQL so you won't have to do this

In [ ]:
# approx_N = df_llr_recs['co_occurrences'].sum()

In [ ]:
# approx_N

In [ ]:
# top_5_df['llr_score'] = top_5_df['llr_score'].map(lambda x: x/approx_N)

In [ ]:
top_5_df.head()

In [ ]:
top_5_df.llr_score.describe()

In [ ]:
top_5_df.co_occurrences.describe()

## Get top 5 links for each page, where co_occurrences > 1, or co_occurrences = 1
Save down these two things to show that co_occurrences = 1 lead to some bad links

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1) ].shape


In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1) & (top_5_df['base_path'] != 'unknown')].shape
#          (top_7_df['co_occurrences'] < 1000)]

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1) & (top_5_df['co_occurrences'] < 10)].shape

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1)][[
    'title', 'link_title', 'base_path', 'link_base_path', 'page', 'link',
     'llr_score','rank','co_occurrences'
    ]].to_csv(
    os.path.join(DATA_DIR, 'llr', 
                 'llr_top_5_coocurrence_gt_1_20190222_20190314.csv'), index=False)

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1) & (top_5_df['base_path']!= 'unknown')][[
    'title', 'link_title', 'base_path', 'link_base_path',
     'llr_score', 'co_occurrences'
    ]].to_csv(
    os.path.join(DATA_DIR, 'llr', 
                 'llr_top_5_coocurrence_gt_1_20190222_20190314_thin.csv'), index=False)

In [ ]:
top_5_df[top_5_df['co_occurrences'] == 1].sort_values('llr_score', ascending=False)[[
    'title', 'link_title', 'base_path', 'link_base_path', 'page', 'link',
     'llr_score','rank','co_occurrences'
    ]].head(100).to_csv(
    os.path.join(DATA_DIR, 'llr', 
                 'llr_top_5_single_coocurrence_highest_scores_20190222_20190314.csv'), index=False)

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1)]['page'].nunique()

In [ ]:
top_5_df[(top_5_df['co_occurrences'] > 1)]['base_path'].nunique()

## Get top 5 links per page for the most popular pages based on some GA data
i.e. get the top 5 links per page for a specified list of base paths (list [here](https://docs.google.com/spreadsheets/d/1s-tyTXaWTv0yLD6WyQDjWfD3UWlvHmgaE_GqSFkYPI4/edit#gid=1914380518)

In [ ]:
top_pages_GA = pd.read_csv(os.path.join(DATA_DIR, 'metadata', 'top_pages.csv'))

In [ ]:
top_pages = top_pages_GA['Page'].tolist()

In [ ]:
def is_top_page(pagepath):
    return pagepath in top_pages

In [ ]:
top_5_df['is_top_page'] = top_5_df['base_path'].map(is_top_page)

In [ ]:
top_5_df[top_5_df['is_top_page']].shape

In [ ]:
top_5_df[top_5_df['is_top_page']][
    [      'title', 'link_title', 'base_path', 'link_base_path', 'page', 'link',
     'llr_score','rank','co_occurrences'
    ]].to_csv(
    os.path.join(DATA_DIR, 'llr', 
                 'llr_top_5_top_GA_pages_20190222_20190314.csv'), index=False)

In [ ]:
top_5_df[top_5_df['is_top_page']]['co_occurrences'].describe()

In [ ]:
top_5_df.columns

## Save top 5 links per page for top GA pages as a JSON file for ingestion into Integration

In [ ]:
top_5_top_GA_pages_dict = {}

for row in top_5_df[top_5_df['is_top_page']].sort_values('llr_score', ascending=False).itertuples():
    if row.page not in top_5_top_GA_pages_dict:
        top_5_top_GA_pages_dict[row.page] = list()
    top_5_top_GA_pages_dict[row.page].append(row.link)

In [ ]:
top_5_top_GA_pages_dict

In [ ]:
ingestion_file = os.path.join(DATA_DIR, 'llr', 'llr_top_5_top_GA_pages_20190222_20190314.json')

In [ ]:
with open(ingestion_file, 'w') as outfile:
    json.dump(top_5_top_GA_pages_dict, outfile)

# A/B Test stuff
Save top 5 links per page for all pages, where co_occurrences > 1 as a JSON file for ingestion for the A/B test


In [ ]:
top_5_all_pages_dict = {}

for row in top_5_df[top_5_df['co_occurrences'] > 1].sort_values(
    'llr_score', ascending=False).itertuples():
    if row[7] not in top_5_all_pages_dict:
        top_5_all_pages_dict[row[7]] = list()
    top_5_all_pages_dict[row[7]].append(row[3])

In [ ]:
ingestion_file_all_pages = os.path.join(DATA_DIR, 'llr', 'llr_top_5_all_pages_20190222_20190314.json')

In [ ]:
with open(ingestion_file_all_pages, 'w') as outfile:
    json.dump(top_5_all_pages_dict, outfile)